# EDA + Feature Engineering for CNN (Unlabeled Screenshots)
Step-by-step notebook covering report sections 1-8. Dataset = 61 user-screenshotted PNGs in `dataset/`, no URLs, no labels. Run top-to-bottom.

In [ ]:
import hashlib, json, re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

RNG = np.random.default_rng(42)
ROOT = Path('dataset')
FIG = Path('outputs/figures')
FIG.mkdir(parents=True, exist_ok=True)
Path('outputs').mkdir(parents=True, exist_ok=True)
TARGET_SMALL = (128, 128)  # for mean-image math (screenshots vary in size)
print('root exists:', ROOT.exists(), '| figures dir:', FIG)

## 1. Dataset card (source, license, size, collection, limitations)

In [ ]:
files = sorted(ROOT.glob('*.png'))
total_bytes = sum(p.stat().st_size for p in files)
modes = Counter(Image.open(p).mode for p in files)
years = []
for p in files:
    m = re.match(r'(\d{4})', p.stem)
    years.append(int(m.group(1)) if m else None)
print(f'n_files = {len(files)}')
print(f'size_on_disk_MB = {total_bytes/1e6:.1f}')
print('modes =', dict(modes))
print('year_range =', min(years), '->', max(years))
print('pre_split = none (flat folder, no train/val/test)')
print('source = user-provided screenshots, no URL')
print('license = private / own screenshots (N/A)')
print('collection = screen-captures; filenames are timestamps, not labels')

## 2. Load & inspect (shape, structure, 5 random records with pixel stats)

In [ ]:
def parse_year(path):
    m = re.match(r'(\d{4})', path.stem)
    return int(m.group(1)) if m else None

recs = []
for p in files:
    im = Image.open(p).convert('RGB')  # drop screenshot alpha channel
    a = np.asarray(im)
    recs.append(dict(path=str(p), year=parse_year(p), width=im.width, height=im.height,
                     mode=Image.open(p).mode, mean=float(a.mean()), std=float(a.std()),
                     min=int(a.min()), max=int(a.max())))
df = pd.DataFrame(recs).sort_values('year').reset_index(drop=True)
print('shape:', df.shape)
print(df.dtypes)
print(df.describe(include='all').round(2))
sample5 = df.sample(5, random_state=42)
print(sample5.to_string(index=True))
df.to_csv('outputs/file_index.csv', index=False)
print('saved outputs/file_index.csv')

## 3. Data quality audit (duplicates, corrupt, degenerate) + action plan

In [ ]:
# exact duplicates via SHA256
hashes = {}
for p in files:
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    hashes.setdefault(h, []).append(p.name)
dup_groups = [v for v in hashes.values() if len(v) > 1]
# corrupt / unreadable
corrupt = []
for p in files:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        corrupt.append((p.name, str(e)))
# degenerate: near-constant (std < 2) or all-black / all-white
degen = df[(df['std'] < 2.0) | ((df['min'] == 0) & (df['max'] == 0)) | ((df['min'] == 255) & (df['max'] == 255))]
print('missing_labels = ALL (unlabeled by design, 0 label files)')
print(f'exact_duplicate_groups = {len(dup_groups)}', dup_groups[:3])
print(f'corrupt_unreadable = {len(corrupt)}', corrupt[:5])
print(f'degenerate = {len(degen)}')
if len(degen): print(degen.to_string())
print('ACTION: drop corrupt (CNN errors on them); keep one copy per duplicate (memorization risk); quarantine degenerate for visual check, drop only if sensor error — justification in report.')

## 4. Class / target distribution (unlabeled: single group + decade pseudo-groups for visualization only)

In [ ]:
df['decade'] = (df['year'] // 10 * 10).astype(str) + 's'
dec_counts = df['decade'].value_counts().sort_index()
print(dec_counts.to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(['unlabeled'], [len(df)], color='steelblue')
ax[0].set_title('Samples: single unlabeled group (n=61)')
ax[0].set_ylabel('count')
dec_counts.plot(kind='bar', ax=ax[1], color='teal')
ax[1].set_title('Pseudo-groups by decade (visualization only, NOT labels)')
ax[1].set_ylabel('count'); plt.xticks(rotation=30)
plt.tight_layout(); plt.savefig(FIG / '01_class_distribution.png', dpi=150); plt.close()
print('saved 01_class_distribution.png | imbalance: N/A (no true classes); accuracy meaningless -> use reconstruction/temporal-consistency metrics if unsupervised, or macro-F1 after labeling')

## 5a. Image size & channel distribution -> target resolution decision

In [ ]:
print('unique_sizes =', df[['width','height']].drop_duplicates().shape[0])
print('mode_of_modes = RGBA -> convert to RGB (drop alpha: screenshots carry useless transparent channel)')
print(df[['width','height']].describe().round(1).to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(df['width'], df['height'], c=df['year'], cmap='viridis')
ax[0].set_xlabel('width'); ax[0].set_ylabel('height'); ax[0].set_title('Size variation (each dot = 1 screenshot)')
ax[1].hist(df['width'], bins=15, alpha=0.7, label='width')
ax[1].hist(df['height'], bins=15, alpha=0.7, label='height')
ax[1].legend(); ax[1].set_title('Width / height histograms')
plt.tight_layout(); plt.savefig(FIG / '02_image_sizes.png', dpi=150); plt.close()
print('saved 02_image_sizes.png')
print('DECISION: target 224x224 RGB (standard CNN input; downscales ~1600x760 screenshots, drops alpha, keeps aspect via resize+center-crop). Justification: 52 unique sizes -> model needs fixed input; alpha carries no signal.')

## 5b. Sample image grid (visual inspection for UI artifacts)

In [ ]:
sel = df.sample(16, random_state=42).sort_values('year')
fig, axes = plt.subplots(4, 4, figsize=(12, 9))
for axi, (_, r) in zip(axes.flat, sel.iterrows()):
    im = Image.open(r['path']).convert('RGB')
    axi.imshow(im); axi.set_title(Path(r['path']).name, fontsize=7); axi.axis('off')
plt.suptitle('16 random screenshots (check for cursors, windows, text overlays)')
plt.tight_layout(); plt.savefig(FIG / '03_samples_grid.png', dpi=150); plt.close()
print('saved 03_samples_grid.png')

## 5c. Pixel intensity histograms (global + per decade)

In [ ]:
def thumb_array(p, size=(256, 256)):
    im = Image.open(p).convert('RGB').resize(size)
    return np.asarray(im)
arrs = np.stack([thumb_array(p) for p in df['path']])
fig, ax = plt.subplots(figsize=(9, 4))
for i, c in enumerate(['r', 'g', 'b']):
    ax.hist(arrs[..., i].ravel(), bins=128, alpha=0.45, color=c, label=c.upper(), density=True)
ax.legend(); ax.set_title('Global RGB intensity (all 61, thumbnailed 256px)'); ax.set_xlabel('0-255')
plt.tight_layout(); plt.savefig(FIG / '04_pixel_histograms.png', dpi=150); plt.close()
print('saved 04_pixel_histograms.png')
print(f'global RGB mean={arrs.mean(axis=(0,1,2)).round(1)} std={arrs.std(axis=(0,1,2)).round(1)}')

## 5d. Mean image per decade + cosine similarity heatmap (which periods look alike?)

In [ ]:
decades = sorted(df['decade'].unique())
means, labels = [], []
for d in decades:
    sub = df[df['decade'] == d]
    if len(sub) == 0: continue
    stack = np.stack([np.asarray(Image.open(p).convert('RGB').resize(TARGET_SMALL), dtype=np.float32) for p in sub['path']])
    means.append(stack.mean(0)); labels.append(f'{d} (n={len(sub)})')
means = np.stack(means)
flat = means.reshape(len(means), -1)
S = cosine_similarity(flat)
fig, axes = plt.subplots(1, len(means), figsize=(3*len(means), 3.2))
if len(means) == 1: axes = [axes]
for axi, m, lb in zip(axes, means, labels):
    axi.imshow(m.astype(np.uint8)); axi.set_title(lb, fontsize=8); axi.axis('off')
plt.suptitle('Mean screenshot per decade'); plt.tight_layout()
plt.savefig(FIG / '05_mean_images.png', dpi=150); plt.close()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(S, vmin=0.9, vmax=1.0, cmap='magma')
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=7); ax.set_yticklabels(labels, fontsize=7)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{S[i,j]:.3f}', ha='center', va='center', fontsize=6, color='white')
plt.colorbar(im, label='cosine similarity'); plt.title('Decade-mean similarity (near-1.0 = look alike)')
plt.tight_layout(); plt.savefig(FIG / '06_mean_similarity_heatmap.png', dpi=150); plt.close()
print('saved 05_mean_images.png, 06_mean_similarity_heatmap.png')
print(pd.DataFrame(S, index=labels, columns=labels).round(3).to_string())

## 6. Train / validation / test split plan (time-ordered, no stratify possible)

In [ ]:
d = df.sort_values('year').reset_index(drop=True)
n = len(d); n_tr, n_va = int(n*0.70), int(n*0.15)
d['split'] = ['train']*n_tr + ['val']*n_va + ['test']*(n - n_tr - n_va)
print(d.groupby('split')['year'].agg(['count','min','max']).to_string())
print('JUSTIFICATION: time-ordered 70/15/15 by filename year avoids temporal leakage (random split would mix eras); no stratify/group IDs exist.')
d[['path','year','decade','split']].to_csv('outputs/split.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 3.5))
for s, c in [('train','steelblue'),('val','orange'),('test','green')]:
    ax.hist(d[d['split']==s]['year'], bins=12, alpha=0.6, label=s, color=c)
ax.legend(); ax.set_title('Time-ordered split by year'); ax.set_xlabel('year')
plt.tight_layout(); plt.savefig(FIG / '07_split.png', dpi=150); plt.close()
print('saved 07_split.png + outputs/split.csv')

## 7-8. Key findings -> preprocessing plan (concrete pipeline + normalization values)

In [ ]:
tr_paths = d[d['split']=='train']['path'].tolist()
tr = np.stack([np.asarray(Image.open(p).convert('RGB').resize((224,224)), dtype=np.float32) for p in tr_paths])
ch_mean = (tr.mean(axis=(0,1,2)) / 255).round(4)
ch_std = (tr.std(axis=(0,1,2)) / 255).round(4)
print('TRAIN RGB mean(0-1):', ch_mean.tolist(), 'std:', ch_std.tolist())
pipe = dict(target_size=[224,224], channels='RGB (RGBA->RGB)', resize='resize+center-crop (52 sizes->fixed)',
             normalize=f'mean={ch_mean.tolist()} std={ch_std.tolist()} (from TRAIN only)',
             augment='small brightness/contrast + slight rotation ONLY; NO h-flip (screenshots may contain text); NO heavy crop (UI edges matter)',
             split='time-ordered 70/15/15', metric='unsupervised: visual-consistency; if labeled later: macro-F1')
print(json.dumps(pipe, indent=2))
json.dump({'n': len(df), 'size_MB': round(sum(Path(p).stat().st_size for p in df['path'])/1e6,1),
           'train_mean': ch_mean.tolist(), 'train_std': ch_std.tolist(), 'pipeline': pipe},
          open('outputs/eda_summary.json','w'), indent=2)
print('saved outputs/eda_summary.json')
print('FINDING 1: all 61 are RGBA w/ 52 sizes -> RGB 224px input (evidence: 5a).')
print('FINDING 2: decade means likely near-identical (see heatmap) -> eras confusable; use time-split + monitor per-era errors (evidence: 5d).')
print('FINDING 3: screenshots risk UI/text artifacts (see grid) -> gentle augmentation only + train-only normalization (evidence: 5b/5c).')